<h1>Bibliotecas</h1>
<i> - Coletar dados de fontes oficiais ligadas ao governo

In [2]:
import requests
import pandas as pd

from datetime import datetime
from pathlib import Path

<h1>Função padrão de exportação RAW</h1>

In [3]:
def salvar_raw(df_exportar, nome_base):
    nome_pipeline = "pipeline_verdadeiro_fontes_oficiais"

    data_agora = datetime.now().strftime("%Y-%m-%d_%H-%M-%S")

    pasta_raw = Path(f"../dados/{nome_pipeline}/raw")
    pasta_raw.mkdir(parents=True, exist_ok=True)

    caminho_saida = pasta_raw / f"{nome_base}_raw_{data_agora}.csv"

    df_exportar.to_csv(
        caminho_saida,
        index=False,
        encoding="utf-8-sig"
    )

    print(f"Arquivo bruto salvo em: {caminho_saida}")
    
    print(f"Total de registros extraídos: {len(df_exportar)}")
    print(f"Data e hora da extração: {datetime.now().strftime('%d/%m/%Y %H:%M:%S')}")

<h1>Coleta de Dados - Sites</h1>

<h2>Camara dos Deputados</h2>

In [4]:
URL_CAMARA = "https://dadosabertos.camara.leg.br/api/v2/proposicoes"

params_camara = {
    "ano": 2026,
    "itens": 20,
    "ordem": "DESC",
    "ordenarPor": "id"
}

resposta_camara = requests.get(URL_CAMARA, params=params_camara)

print(resposta_camara.status_code)

dados_camara = resposta_camara.json()

proposicoes = dados_camara.get("dados", [])

df_camara_raw = pd.DataFrame(proposicoes)

df_camara_raw.head()

salvar_raw(df_camara_raw, "camara_proposicoes")

200
Arquivo bruto salvo em: ..\dados\pipeline_verdadeiro_fontes_oficiais\raw\camara_proposicoes_raw_2026-05-10_02-18-24.csv
Total de registros extraídos: 20
Data e hora da extração: 10/05/2026 02:18:24


<h2>Senado</h2>

In [13]:
URL_SENADO = "https://legis.senado.leg.br/dadosabertos/materia/pesquisa/lista.json"

params_senado = {
    "ano": 2026
}

resposta_senado = requests.get(URL_SENADO, params=params_senado)

print(resposta_senado.status_code)
print(resposta_senado.url)

dados_senado = resposta_senado.json()

dados_senado.keys()

materias = dados_senado.get("PesquisaBasicaMateria", {}).get("Materias", {}).get("Materia", [])

df_senado_raw = pd.DataFrame(materias)

df_senado_raw.head()

df_senado_raw["fonte_verificacao"] = "SENADO_FEDERAL"
df_senado_raw["url_consulta"] = resposta_senado.url
df_senado_raw["data_coleta"] = datetime.now().strftime("%d/%m/%Y %H:%M:%S")

salvar_raw(df_senado_raw, "senado_materias")

200
https://legis.senado.leg.br/dadosabertos/materia/pesquisa/lista.json?ano=2026
Arquivo bruto salvo em: ..\dados\pipeline_verdadeiro_fontes_oficiais\raw\senado_materias_raw_2026-05-10_02-28-13.csv
Total de registros extraídos: 1381
Data e hora da extração: 10/05/2026 02:28:13


<h2>TSE</h2>

In [ ]:
# site do TSE está em manutenção, validar futuramente...